# 전처리
`Descriptive_Statistic_v3.ipynb`에서 확인한 점검 및 기술통계 결과를 바탕으로 `steam_indie_9692` 를 전처리함

# 1. 라이브러리 호출

In [64]:
from pathlib import Path
import ast
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# 한글 폰트 설정
# Windows에서는 Malgun Gothic, macOS에서는 AppleGothic, 그 외 환경에서는 NanumGothic을 사용합니다.
import platform

if platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
elif platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
else:
    plt.rcParams["font.family"] = "NanumGothic"

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (12, 6)

# pandas 출력 옵션
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 180)

# 2. 파일 경로 설정 및 데이터 읽기

In [65]:
# 프로젝트 루트 직접 지정
# 다른 환경에서 실행할 경우 ROOT만 본인 프로젝트 경로에 맞게 수정합니다.
ROOT = Path(r"C:\Users\joon5\Documents\github\steam-indie-game-analysis")

# 원천/소스 파일이 들어있는 폴더
DATA_DIR = ROOT / "data" / "processed"

# 본 분석 메인 데이터 파일
MAIN_PATH = DATA_DIR / "steam_indie_9692_202604281029.csv"

# 전처리 결과 저장 경로 설정
# 필요하면 폴더명만 바꿔도 됩니다.
OUTPUT_DIR = ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 저장 파일명
PREPROCESSED_PATH = OUTPUT_DIR / "steam_indie_9692_preprocessed_v1.csv"
GENRE_EXPLODED_PATH = OUTPUT_DIR / "steam_indie_9692_genre_exploded_v1.csv"

# 경로 확인
print("ROOT                =", ROOT)
print("DATA_DIR            =", DATA_DIR)
print("MAIN_PATH           =", MAIN_PATH)
print("PREPROCESSED_PATH   =", PREPROCESSED_PATH)
print("GENRE_EXPLODED_PATH =", GENRE_EXPLODED_PATH)
print()
print("main exists:", MAIN_PATH.exists())

ROOT                = C:\Users\joon5\Documents\github\steam-indie-game-analysis
DATA_DIR            = C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\processed
MAIN_PATH           = C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\processed\steam_indie_9692_202604281029.csv
PREPROCESSED_PATH   = C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\processed\steam_indie_9692_preprocessed_v1.csv
GENRE_EXPLODED_PATH = C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\processed\steam_indie_9692_genre_exploded_v1.csv

main exists: True


In [66]:
# 원본 데이터 읽기
# df_raw는 원본 보존용, 실제 전처리는 df_indie에서 진행
df_raw = pd.read_csv(MAIN_PATH)
df_indie = df_raw.copy()


print("df_raw shape   :", df_raw.shape)
print("df_indie shape :", df_indie.shape)

display(df_indie.head())

df_raw shape   : (9692, 14)
df_indie shape : (9692, 14)


,appid,name,owners,positive,negative,price,ccu,genres,release_date,developers,total_reviews,owners_lower,is_f2p,is_early_access
0,899770,Last Epoch,"20,000,000 .. 50,000,000",88027,22596,3499,5831,"['Action', 'Adventure', 'Indie', 'RPG']",2024-02-21,Eleventh Hour Games,110623,20000000,False,False
1,251570,7 Days to Die,"10,000,000 .. 20,000,000",327889,42157,4499,17045,"['Action', 'Adventure', 'Indie', 'RPG', 'Simul...",2024-07-25,The Fun Pimps,370046,10000000,False,False
2,1116170,CyberCorp,"10,000,000 .. 20,000,000",266,56,1499,3,"['Action', 'Adventure', 'Indie', 'RPG']",2025-04-22,Megame LLC,322,10000000,False,False
3,1326470,Sons Of The Forest,"10,000,000 .. 20,000,000",222495,31051,2999,4450,"['Action', 'Adventure', 'Indie', 'Simulation']",2024-02-22,Endnight Games Ltd,253546,10000000,False,False
4,2186680,"Warhammer 40,000: Rogue Trader","10,000,000 .. 20,000,000",26360,4445,4999,3582,"['Action', 'Adventure', 'Indie', 'RPG', 'Strat...",2023-12-07,Owlcat Games,30805,10000000,False,False


# 3. 전처리 전 기본 확인

In [67]:
def check_basic_info(df, df_name, exclude_cols=None):
    """행/열 수, 완전 중복 행, 컬럼별 타입/결측/고유값을 한 번에 확인"""
    print(f"\n{'='*80}")
    print(f"{df_name}의 기본 정보 / 타입 / 결측치 확인")
    print(f"{'='*80}\n")

    # 원본을 건드리지 않기 위해 복사본에서만 확인
    df_copied = df.copy()
    
    # 제외할 컬럼 반영
    if exclude_cols:
        df_copied = df_copied.drop(columns=exclude_cols, errors='ignore')

    # dict, list, set 같은 해시 불가능 값이 들어있는 컬럼은 문자열로 변환
    for col in df_copied.columns:
        try:
            df_copied[col].nunique(dropna=True)
        except TypeError:
            df_copied[col] = df_copied[col].astype(str)

    # 전체 요약
    overview_df = pd.DataFrame({
        '항목': ['행 개수', '열 개수', '중복 행 개수'],
        '값': [df_copied.shape[0], df_copied.shape[1], df_copied.duplicated().sum()]
    })

    # 컬럼별 요약
    summary_df = pd.DataFrame({
        '데이터타입': df_copied.dtypes.astype(str),
        '행 개수': df_copied.count(),
        '행 비율(%)': (df_copied.count() / len(df_copied) * 100).round(2),
        '결측치 개수': df_copied.isnull().sum(),
        '결측치 비율(%)': (df_copied.isnull().sum() / len(df_copied) * 100).round(2),
        '고유값 개수': df_copied.nunique(dropna=True)
    }).sort_values(by=['결측치 개수', '고유값 개수'], ascending=[False, False])

    print("[전체 요약]")
    display(overview_df)

    print("[컬럼별 요약]")
    display(summary_df)

    print("[상위 5행]")
    display(df_copied.head())

In [68]:
check_basic_info(df_indie, "df_indie")


df_indie의 기본 정보 / 타입 / 결측치 확인

[전체 요약]


,항목,값
0,행 개수,9692
1,열 개수,14
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
developers,str,9680,99.88,12,0.12,8139
appid,int64,9692,100.00,0,0.00,9692
name,str,9692,100.00,0,0.00,9688
total_reviews,int64,9692,100.00,0,0.00,1427
positive,int64,9692,100.00,0,0.00,1346
release_date,str,9692,100.00,0,0.00,1008
negative,int64,9692,100.00,0,0.00,601
ccu,int64,9692,100.00,0,0.00,294
price,int64,9692,100.00,0,0.00,282
genres,str,9692,100.00,0,0.00,250


[상위 5행]


,appid,name,owners,positive,negative,price,ccu,genres,release_date,developers,total_reviews,owners_lower,is_f2p,is_early_access
0,899770,Last Epoch,"20,000,000 .. 50,000,000",88027,22596,3499,5831,"['Action', 'Adventure', 'Indie', 'RPG']",2024-02-21,Eleventh Hour Games,110623,20000000,False,False
1,251570,7 Days to Die,"10,000,000 .. 20,000,000",327889,42157,4499,17045,"['Action', 'Adventure', 'Indie', 'RPG', 'Simul...",2024-07-25,The Fun Pimps,370046,10000000,False,False
2,1116170,CyberCorp,"10,000,000 .. 20,000,000",266,56,1499,3,"['Action', 'Adventure', 'Indie', 'RPG']",2025-04-22,Megame LLC,322,10000000,False,False
3,1326470,Sons Of The Forest,"10,000,000 .. 20,000,000",222495,31051,2999,4450,"['Action', 'Adventure', 'Indie', 'Simulation']",2024-02-22,Endnight Games Ltd,253546,10000000,False,False
4,2186680,"Warhammer 40,000: Rogue Trader","10,000,000 .. 20,000,000",26360,4445,4999,3582,"['Action', 'Adventure', 'Indie', 'RPG', 'Strat...",2023-12-07,Owlcat Games,30805,10000000,False,False


In [69]:
def check_id_duplicates(df, col_name, df_name, top_n=10):
    """기준 키로 쓸 컬럼의 중복 여부를 확인"""
    print(f"\n{'='*80}")
    print(f"{df_name}의 {col_name} 값 중복 확인")
    print(f"{'='*80}")

    df_copied = df.copy()

    if col_name not in df_copied.columns:
        print(f"'{col_name}' 컬럼이 존재하지 않습니다.")
        return

    duplicate_count = df_copied[col_name].duplicated().sum()

    print('전체 행 수:', len(df_copied))
    print(f'{col_name} 고유 개수:', df_copied[col_name].nunique(dropna=True))
    print(f'중복 {col_name} 개수:', duplicate_count)

    if duplicate_count > 0:
        print()
        print('[중복 상위 값]')
        dup_summary = df_copied[col_name].value_counts(dropna=False).reset_index()
        dup_summary.columns = [col_name, '등장 횟수']
        display(dup_summary[dup_summary['등장 횟수'] > 1].head(top_n))
    else:
        print('중복 값이 없습니다.')

In [70]:
check_id_duplicates(df_indie, "appid", "df_indie")


df_indie의 appid 값 중복 확인
전체 행 수: 9692
appid 고유 개수: 9692
중복 appid 개수: 0
중복 값이 없습니다.


# 전처리

## 1. 문자열 컬럼 공백 정리

In [71]:
# 공백 정리 전 확인할 문자열 컬럼 목록
text_cols = ["name", "developers", "owners", "release_date", "genres"]

space_before_rows = []

for col in text_cols:
    if col in df_indie.columns:
        # 결측치를 제외하고 문자열로 변환한 뒤 공백 상태를 확인
        s = df_indie[col].dropna().astype(str)

        space_before_rows.append({
            "column": col,
            "blank_count_before": s.str.strip().eq("").sum(),
            "leading_trailing_space_before": s.ne(s.str.strip()).sum(),
            "multi_space_before": s.str.contains(r"\s{2,}", regex=True).sum()
        })

space_before_df = pd.DataFrame(space_before_rows)
display(space_before_df)

,column,blank_count_before,leading_trailing_space_before,multi_space_before
0,name,0,7,17
1,developers,0,11,5
2,owners,0,0,0
3,release_date,0,0,0
4,genres,0,0,0


In [72]:
# 문자열 공백 정리
for col in ["name", "developers"]:
    if col in df_indie.columns:
        df_indie[col] = (
            df_indie[col]
            .astype("string")
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
        )

# owners, release_date는 원본 구조를 최대한 유지하면서 앞뒤 공백만 정리
for col in ["owners", "release_date"]:
    if col in df_indie.columns:
        df_indie[col] = df_indie[col].astype("string").str.strip()

# genres는 문자열 리스트 구조이므로 전체 앞뒤 공백만 제거
if "genres" in df_indie.columns:
    df_indie["genres"] = df_indie["genres"].astype("string").str.strip()

In [73]:
# 문자열 공백 정리 후 확인
space_after_rows = []

for col in text_cols:
    if col in df_indie.columns:
        s = df_indie[col].dropna().astype(str)

        space_after_rows.append({
            "column": col,
            "blank_count_after": s.str.strip().eq("").sum(),
            "leading_trailing_space_after": s.ne(s.str.strip()).sum(),
            "multi_space_after": s.str.contains(r"\s{2,}", regex=True).sum()
        })

space_after_df = pd.DataFrame(space_after_rows)
display(space_after_df)

,column,blank_count_after,leading_trailing_space_after,multi_space_after
0,name,0,0,0
1,developers,0,0,0
2,owners,0,0,0
3,release_date,0,0,0
4,genres,0,0,0


## 2. 출시일 컬럼 변환

In [74]:
# release_date를 datetime으로 변환
# errors="coerce"는 변환할 수 없는 날짜를 NaT로 처리
df_indie["release_date_dt"] = pd.to_datetime(df_indie["release_date"], errors="coerce")

# 날짜 분석에 자주 쓰는 파생 컬럼을 생성
df_indie["release_year"] = df_indie["release_date_dt"].dt.year
df_indie["release_month"] = df_indie["release_date_dt"].dt.month
df_indie["release_quarter"] = df_indie["release_date_dt"].dt.to_period("Q").astype("string")

print("원본 release_date 결측 수:", df_indie["release_date"].isna().sum())
print("release_date_dt 변환 실패 수:", df_indie["release_date_dt"].isna().sum())
print("최소 출시일:", df_indie["release_date_dt"].min())
print("최대 출시일:", df_indie["release_date_dt"].max())

print()
print("연도별 게임 수")

release_year_summary = (
    df_indie["release_year"]
    .value_counts(dropna=False)
    .sort_index()
    .reset_index()
)

release_year_summary.columns = ["release_year", "game_count"]
display(release_year_summary)

원본 release_date 결측 수: 0
release_date_dt 변환 실패 수: 0
최소 출시일: 2023-01-01 00:00:00
최대 출시일: 2025-12-27 00:00:00

연도별 게임 수


,release_year,game_count
0,2023,3498
1,2024,4180
2,2025,2014


## 3. 장르 컬럼 파싱

In [75]:
# genres 문자열을 실제 list로 변환
genres_list = []
genre_parse_failed = []

for value in df_indie["genres"]:

    try:
        # 이미 리스트인 경우 그대로 사용
        if isinstance(value, list):
            parsed = value

        # 결측값이면 빈 리스트로 처리
        elif pd.isna(value):
            parsed = []

        # 문자열 형태의 리스트면 ast.literal_eval로 안전하게 파싱
        else:
            parsed = ast.literal_eval(str(value))

        # 파싱 결과가 리스트인지 확인합니다.
        if isinstance(parsed, list):
            # 각 장르명도 문자열로 변환하고 앞뒤 공백을 제거합니다.
            parsed = [str(x).strip() for x in parsed if str(x).strip() != ""]
            genres_list.append(parsed)
            genre_parse_failed.append(False)
        else:
            # 리스트가 아닌 형태로 파싱되면 실패로 기록
            genres_list.append([])
            genre_parse_failed.append(True)

    except Exception:
        # 파싱 자체가 실패하면 빈 리스트로 넣고 실패 여부 기록
        genres_list.append([])
        genre_parse_failed.append(True)


# 파싱 결과 컬럼 생성
df_indie["genres_list"] = genres_list
df_indie["genre_count"] = df_indie["genres_list"].apply(len)
df_indie["genre_parse_failed"] = genre_parse_failed

print("genres 파싱 실패 수:", df_indie["genre_parse_failed"].sum())
print("평균 장르 수:", round(df_indie["genre_count"].mean(), 2))
print("최소 장르 수:", df_indie["genre_count"].min())
print("최대 장르 수:", df_indie["genre_count"].max())

print()
print("장르 개수 분포")
genre_count_summary = (
    df_indie["genre_count"]
    .value_counts()
    .sort_index()
    .reset_index()
)
genre_count_summary.columns = ["genre_count", "game_count"]
display(genre_count_summary)

genres 파싱 실패 수: 0
평균 장르 수: 3.12
최소 장르 수: 1
최대 장르 수: 10

장르 개수 분포


,genre_count,game_count
0,1,371
1,2,2597
2,3,3698
3,4,1979
4,5,755
5,6,212
6,7,47
7,8,16
8,9,15
9,10,2


In [76]:
# 장르별 분석을 위해 game 단위 데이터를 장르 단위로 펼친 별도 테이블 생성
# 한 게임이 여러 장르를 가질 수 있으므로, genre_df는 appid가 중복될 수 있다.
genre_df = (
    df_indie[["appid", "name", "genres_list"]]
    .explode("genres_list")
    .rename(columns={"genres_list": "genre"})
)

# 장르가 비어 있는 행은 제거
genre_df = genre_df[genre_df["genre"].notna()].copy()

print("genre_df shape:", genre_df.shape)
display(genre_df.head())

print()
print("장르별 게임 수")

genre_summary = (
    genre_df["genre"]
    .value_counts()
    .reset_index()
)

genre_summary.columns = ["genre", "game_count"]
genre_summary["ratio_pct"] = (genre_summary["game_count"] / len(df_indie) * 100).round(2)

display(genre_summary)

genre_df shape: (30234, 3)


,appid,name,genre
0,899770,Last Epoch,Action
0,899770,Last Epoch,Adventure
0,899770,Last Epoch,Indie
0,899770,Last Epoch,RPG
1,251570,7 Days to Die,Action



장르별 게임 수


,genre,game_count,ratio_pct
0,Indie,9687,99.95
1,Adventure,4807,49.60
2,Casual,4111,42.42
3,Action,4093,42.23
4,Simulation,2503,25.83
5,RPG,2194,22.64
6,Strategy,2005,20.69
7,Sports,341,3.52
8,Racing,301,3.11
9,Massively Multiplayer,124,1.28


## 4. 리뷰 수 및 긍정률 파생 컬럼

In [77]:
# positive + negative가 total_reviews와 일치하는지 확인
# total_reviews가 이미 원본 데이터에 존재하더라도 검증용으로 다시 계산
df_indie["total_reviews_calc"] = df_indie["positive"] + df_indie["negative"]
df_indie["total_reviews_match"] = df_indie["total_reviews_calc"].eq(df_indie["total_reviews"])

mismatch_count = (~df_indie["total_reviews_match"]).sum()
print("total_reviews 불일치 행 수:", mismatch_count)

if mismatch_count > 0:
    display(df_indie.loc[~df_indie["total_reviews_match"], [
        "appid", 
        "name", 
        "positive", 
        "negative", 
        "total_reviews",
        "total_reviews_calc"
    ]].head(20))

total_reviews 불일치 행 수: 0


In [78]:
# 긍정률 계산
# total_reviews가 0인 경우는 np.nan으로 처리
df_indie["positive_ratio"] = np.where(
    df_indie["total_reviews"] > 0,
    df_indie["positive"] / df_indie["total_reviews"],
    np.nan
)

# 보고서와 시각화에서 보기 쉽도록 백분율 컬럼 생성합
df_indie["positive_ratio_pct"] = (df_indie["positive_ratio"] * 100).round(2)

print("positive_ratio_pct 기술통계")
display(df_indie["positive_ratio_pct"].describe().to_frame().T.round(2))

print()
print("긍정률 상위 게임")

display(df_indie[
        [
            "appid", 
            "name", 
            "total_reviews", 
            "positive", 
            "negative", 
            "positive_ratio_pct"
        ]
            ]
        .sort_values(["positive_ratio_pct", "total_reviews"], ascending=[False, False])
        .head(10))

positive_ratio_pct 기술통계


,count,mean,std,min,25%,50%,75%,max
positive_ratio_pct,9692.0,84.18,15.2,0.0,77.06,88.37,95.34,100.0



긍정률 상위 게임


,appid,name,total_reviews,positive,negative,positive_ratio_pct
2214,3247500,"Shooters, Ready!",229,229,0,100.0
1134,2391870,MareQuest: An Interactive Tail,190,190,0,100.0
1713,3035990,Misericorde Volume Two: White Wool & Snow,175,175,0,100.0
1144,2769210,Scarmonde,163,163,0,100.0
7307,3429910,Little Adventurer Treasure Hunt,152,152,0,100.0
6181,1068460,Stuffo the Puzzle Bot,143,143,0,100.0
5671,2677040,真夜的居所 - Chanye's Home,141,141,0,100.0
2968,1707400,Larcin Lazer,123,123,0,100.0
2436,3238510,Fledgling Manor,116,116,0,100.0
4041,2429100,Homestar Runner: Halloween Hide n' Seek,114,114,0,100.0


# 전처리 후 최종 확인

In [86]:
check_basic_info(df_indie, "steam_indie")
check_id_duplicates(df_indie, "appid", "steam_indie")


steam_indie의 기본 정보 / 타입 / 결측치 확인

[전체 요약]


,항목,값
0,행 개수,9692
1,열 개수,25
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
developers,string,9680,99.88,12,0.12,8136
appid,int64,9692,100.00,0,0.00,9692
name,string,9692,100.00,0,0.00,9688
positive_ratio,float64,9692,100.00,0,0.00,3375
positive_ratio_pct,float64,9692,100.00,0,0.00,2462
total_reviews,int64,9692,100.00,0,0.00,1427
total_reviews_calc,int64,9692,100.00,0,0.00,1427
positive,int64,9692,100.00,0,0.00,1346
release_date,string,9692,100.00,0,0.00,1008
release_date_dt,datetime64[us],9692,100.00,0,0.00,1008


[상위 5행]


,appid,name,owners,positive,negative,price,ccu,genres,release_date,developers,total_reviews,owners_lower,is_f2p,is_early_access,release_date_dt,release_year,release_month,release_quarter,genres_list,genre_count,genre_parse_failed,total_reviews_calc,total_reviews_match,positive_ratio,positive_ratio_pct
0,899770,Last Epoch,"20,000,000 .. 50,000,000",88027,22596,3499,5831,"['Action', 'Adventure', 'Indie', 'RPG']",2024-02-21,Eleventh Hour Games,110623,20000000,False,False,2024-02-21,2024,2,2024Q1,"['Action', 'Adventure', 'Indie', 'RPG']",4,False,110623,True,0.795739,79.57
1,251570,7 Days to Die,"10,000,000 .. 20,000,000",327889,42157,4499,17045,"['Action', 'Adventure', 'Indie', 'RPG', 'Simul...",2024-07-25,The Fun Pimps,370046,10000000,False,False,2024-07-25,2024,7,2024Q3,"['Action', 'Adventure', 'Indie', 'RPG', 'Simul...",6,False,370046,True,0.886076,88.61
2,1116170,CyberCorp,"10,000,000 .. 20,000,000",266,56,1499,3,"['Action', 'Adventure', 'Indie', 'RPG']",2025-04-22,Megame LLC,322,10000000,False,False,2025-04-22,2025,4,2025Q2,"['Action', 'Adventure', 'Indie', 'RPG']",4,False,322,True,0.826087,82.61
3,1326470,Sons Of The Forest,"10,000,000 .. 20,000,000",222495,31051,2999,4450,"['Action', 'Adventure', 'Indie', 'Simulation']",2024-02-22,Endnight Games Ltd,253546,10000000,False,False,2024-02-22,2024,2,2024Q1,"['Action', 'Adventure', 'Indie', 'Simulation']",4,False,253546,True,0.877533,87.75
4,2186680,"Warhammer 40,000: Rogue Trader","10,000,000 .. 20,000,000",26360,4445,4999,3582,"['Action', 'Adventure', 'Indie', 'RPG', 'Strat...",2023-12-07,Owlcat Games,30805,10000000,False,False,2023-12-07,2023,12,2023Q4,"['Action', 'Adventure', 'Indie', 'RPG', 'Strat...",5,False,30805,True,0.855705,85.57



steam_indie의 appid 값 중복 확인
전체 행 수: 9692
appid 고유 개수: 9692
중복 appid 개수: 0
중복 값이 없습니다.


In [88]:
check_basic_info(genre_df, "genre_df")


genre_df의 기본 정보 / 타입 / 결측치 확인

[전체 요약]


,항목,값
0,행 개수,30234
1,열 개수,3
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
appid,int64,30234,100.0,0,0.0,9692
name,string,30234,100.0,0,0.0,9688
genre,str,30234,100.0,0,0.0,21


[상위 5행]


,appid,name,genre
0,899770,Last Epoch,Action
0,899770,Last Epoch,Adventure
0,899770,Last Epoch,Indie
0,899770,Last Epoch,RPG
1,251570,7 Days to Die,Action


In [84]:
print()
print("전처리 후 컬럼 목록")
column_dtype_after = pd.DataFrame({
    "column": df_indie.columns, 
    "dtype": df_indie.dtypes.astype(str).values
})
display(column_dtype_after)


전처리 후 컬럼 목록


,column,dtype
0,appid,int64
1,name,string
2,owners,string
3,positive,int64
4,negative,int64
5,price,int64
6,ccu,int64
7,genres,string
8,release_date,string
9,developers,string


# 전처리 결과 저장

In [85]:
df_indie.to_csv(PREPROCESSED_PATH, index=False, encoding="utf-8-sig")
genre_df.to_csv(GENRE_EXPLODED_PATH, index=False, encoding="utf-8-sig")

print("전처리 메인 데이터 저장 완료:", PREPROCESSED_PATH)           # 본 분석용 전처리 메인 데이터
print("장르 explode 데이터 저장 완료:", GENRE_EXPLODED_PATH)        # 장르별 분석을 위한 explode 결과 데이터

전처리 메인 데이터 저장 완료: C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\processed\steam_indie_9692_preprocessed_v1.csv
장르 explode 데이터 저장 완료: C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\processed\steam_indie_9692_genre_exploded_v1.csv


# 최종 컬럼 한 줄 설명

## 원본 컬럼

| 컬럼명 | 한 줄 설명 |
|---|---|
| `appid` | Steam 게임 고유 ID이며, 게임 단위 식별자이다. |
| `name` | Steam 게임명이다. |
| `owners` | SteamSpy 기준 소유자 수 추정 범위 문자열이다. |
| `positive` | 누적 긍정 리뷰 수이다. |
| `negative` | 누적 부정 리뷰 수이다. |
| `price` | 원본 데이터에 수집된 가격 값이다. 단위는 분석 전 명세 확인이 필요하다. |
| `ccu` | 수집 시점 기준 최고 동시접속자 수 지표이다. |
| `genres` | 문자열 형태로 저장된 장르 리스트이다. |
| `release_date` | 게임 출시일 원본 문자열이다. |
| `developers` | 개발사명이다. |
| `total_reviews` | 전체 리뷰 수이며, 일반적으로 `positive + negative`와 일치해야 한다. |
| `owners_lower` | `owners` 범위의 하한값이다. |
| `is_f2p` | 무료 게임 여부를 나타내는 컬럼이다. |
| `is_early_access` | 얼리 액세스 게임 여부를 나타내는 컬럼이다. |

## 전처리로 추가한 컬럼

| 컬럼명 | 한 줄 설명 |
|---|---|
| `release_date_dt` | `release_date`를 datetime 형식으로 변환한 출시일 컬럼이다. |
| `release_year` | 출시 연도이다. |
| `release_month` | 출시 월이다. |
| `release_quarter` | 출시 분기이다. |
| `genres_list` | `genres` 문자열을 실제 리스트로 변환한 컬럼이다. |
| `genre_count` | 한 게임에 부여된 장르 개수이다. |
| `genre_parse_failed` | `genres` 파싱 실패 여부이다. |
| `total_reviews_calc` | `positive + negative`로 다시 계산한 전체 리뷰 수이다. |
| `total_reviews_match` | 원본 `total_reviews`와 계산값이 일치하는지 여부이다. |
| `positive_ratio` | 전체 리뷰 중 긍정 리뷰 비율이다. |
| `positive_ratio_pct` | 긍정 리뷰 비율을 퍼센트로 표현한 값이다. |